# PhoBERT Fine-tune V4 - Advanced Techniques

## Mục Tiêu
Fine-tune PhoBERT với dữ liệu balanced (data_cleaned_finetune.csv) sử dụng các kỹ thuật advanced để đạt accuracy > 90%.

## Nguồn Dữ Liệu
- **File**: data_v4_tu_crawl/data_cleaned_finetune.csv (30,000 samples)
- **Đặc điểm**: Balanced dữ liệu cho 3 classes (Negative, Positive, Neutral)

## Các Kỹ Thuật Advanced
- **CLS + Mean Pooling**: Kết hợp [CLS] token (768) + mean pooling (768) = 1536 dim
- **MLP Classifier Head**: Linear(1536→512, GELU) + MultiDropout(5x) + Linear(512→3)
- **FGM Adversarial**: Perturbation embeddings (eps=0.5) mỗi step
- **AWP (Adversarial Weight Perturbation)**: Full strength từ epoch 2+
- **Focal Loss** (γ=2.5) + Label Smoothing (0.1)
- **R-Drop** (α=5.0): Regularization giữa 2 forward passes
- **EMA** (decay=0.999): Exponential Moving Average cho model weights
- **LLRD** (decay=0.95): Layer-wise Learning Rate Decay
- **Gradient Checkpointing**: Giảm VRAM, tăng batch size

## Optimization
- Tokenization cache trên Drive
- Dynamic padding + pad_to_multiple_of=8
- Fused AdamW
- Group by length
- Early stopping (patience=5)
- Resume training từ checkpoint nếu bị ngắt

## Hiệu Suất Dự Kiến
V1: 75% → V2: 80% → V3: 95% → V4: >90% (balanced data)


In [ ]:
!pip install -q transformers==4.44.2 datasets accelerate scikit-learn seaborn matplotlib

In [ ]:
import torch, sys, os, json, glob, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoConfig, TrainingArguments, Trainer,
    EarlyStoppingCallback, TrainerCallback,
    RobertaModel, RobertaPreTrainedModel,
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

LABEL_NAMES = {0: 'NEGATIVE', 1: 'POSITIVE', 2: 'NEUTRAL'}
NUM_LABELS  = 3
MODEL_NAME  = 'vinai/phobert-base-v2'
MAX_LENGTH  = 256
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not torch.cuda.is_available():
    print('No GPU detected. Go to Runtime -> Change runtime type -> GPU')
    sys.exit(1)

gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU    : {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR         = '/content/drive/MyDrive/DO_AN_1_main'
DATA_FILE        = f'{BASE_DIR}/data/data_v4_tu_crawl/data_cleaned_finetune.csv'
OUTPUT_DIR       = f'{BASE_DIR}/phobert-v4'
CHECKPOINT_DIR   = f'{BASE_DIR}/phobert-v4-checkpoint'
TOKENIZED_CACHE  = f'{CHECKPOINT_DIR}/tokenized_cache'
EMA_STATE_FILE   = f'{CHECKPOINT_DIR}/ema_state.pt'
METRICS_LOG_FILE = f'{CHECKPOINT_DIR}/metrics_log.json'

for d in [OUTPUT_DIR, CHECKPOINT_DIR, TOKENIZED_CACHE]:
    os.makedirs(d, exist_ok=True)

print(f'Data      : {DATA_FILE}')
print(f'Exists    : {os.path.exists(DATA_FILE)}')
print(f'Output    : {OUTPUT_DIR}')
print(f'Checkpoint: {CHECKPOINT_DIR}')

In [ ]:
df      = pd.read_csv(DATA_FILE)
col_map = {c.lower(): c for c in df.columns}
df = df.rename(columns={
    col_map.get('review',       col_map.get('text',         'text')):  'text',
    col_map.get('label_number', col_map.get('label_number', 'label')): 'label'
})
df          = df[['text', 'label']].dropna()
df['text']  = df['text'].astype(str).str.strip()
df          = df[df['text'].str.len() > 0].reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f'Loaded: {len(df):,} samples')
for lbl, name in LABEL_NAMES.items():
    n = (df['label'] == lbl).sum()
    print(f'  {name}: {n:,} ({n/len(df)*100:.1f}%)')

df_train, df_temp = train_test_split(df,      test_size=0.2, random_state=SEED, stratify=df['label'])
df_val,   df_test = train_test_split(df_temp, test_size=0.5, random_state=SEED, stratify=df_temp['label'])
print(f'Split -> Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')

class_weights        = compute_class_weight('balanced', classes=np.array([0,1,2]), y=np.array(df_train['label']))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f'Class weights: {[f"{w:.4f}" for w in class_weights]}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH)

def make_dataset(df_, split_name):
    cache_path = os.path.join(TOKENIZED_CACHE, split_name)
    if os.path.exists(cache_path):
        from datasets import load_from_disk
        ds = load_from_disk(cache_path)
        print(f'  {split_name}: loaded from cache ({len(ds):,} samples)')
    else:
        ds = Dataset.from_pandas(df_[['text', 'label']].reset_index(drop=True))
        ds = ds.map(tokenize_fn, batched=True, batch_size=1000, num_proc=2)
        ds.save_to_disk(cache_path)
        print(f'  {split_name}: tokenized and cached ({len(ds):,} samples)')
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds

print('Tokenizing (su dung cache neu co):')
tk_train = make_dataset(df_train, 'train')
tk_val   = make_dataset(df_val,   'val')
tk_test  = make_dataset(df_test,  'test')

# Dynamic padding: pad theo batch, align 8 cho Tensor Core
data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
print(f'Ready -> Train: {len(tk_train):,} | Val: {len(tk_val):,} | Test: {len(tk_test):,}')

In [ ]:
class PhoBERTAdvanced(RobertaPreTrainedModel):
    """
    Nang cap so voi phien ban cu:

    1. CLS + Mean Pooling:
       - [CLS] token (768 dim): bieu dien global cua ca cau
       - Mean pooling (768 dim): trung binh co trong cua tat ca token
       - Concatenate -> 1536 dim: phong phu hon ~0.5-1% accuracy

    2. MLP Classifier Head voi GELU:
       - Linear(1536 -> 512) + GELU + Dropout
       - Linear(512 -> 3)
       - Phi tuyen (GELU) giup hoc pattern phuc tap hon linear don gian

    3. Multi-Sample Dropout ap dung o layer trung gian:
       - 5 dropout masks tren hidden layer 512 dim
       - Regularization manh hon
    """

    def __init__(self, config, num_samples=5, dropout_rate=0.3):
        super().__init__(config)
        self.num_labels  = config.num_labels
        self.num_samples = num_samples
        self.roberta     = RobertaModel(config, add_pooling_layer=False)

        hidden = config.hidden_size          # 768
        pool   = hidden * 2                  # 1536 (CLS + mean)
        mid    = 512

        # Project pooled vector xuong 512
        self.proj     = nn.Linear(pool, mid)
        self.act      = nn.GELU()
        self.dropouts = nn.ModuleList([nn.Dropout(dropout_rate) for _ in range(num_samples)])
        self.out_proj = nn.Linear(mid, config.num_labels)
        self.post_init()

    def _pool(self, hidden_states, attention_mask):
        cls_vec  = hidden_states[:, 0]                                           # [B, 768]
        mask_exp = attention_mask.unsqueeze(-1).float()                          # [B, L, 1]
        mean_vec = (hidden_states * mask_exp).sum(1) / mask_exp.sum(1).clamp(1) # [B, 768]
        return torch.cat([cls_vec, mean_vec], dim=-1)                            # [B, 1536]

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        hidden = self.roberta(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state                                    # [B, L, 768]

        pooled = self._pool(hidden, attention_mask)            # [B, 1536]
        h      = self.act(self.proj(pooled))                   # [B, 512]

        if self.training:
            # Multi-Sample Dropout tren hidden 512
            logits = torch.stack([self.out_proj(d(h)) for d in self.dropouts]).mean(0)
        else:
            logits = self.out_proj(h)

        loss = nn.CrossEntropyLoss()(
            logits.view(-1, self.num_labels), labels.view(-1)
        ) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)


config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

# Gradient checkpointing: CUDA luu lai activation theo yeu cau thay vi giu het trong VRAM
# Giam VRAM ~30%, cho phep batch_size=32 chay on dinh tren T4 16GB
# Toc do cham hon ~10% nhung doi lai VRAM de tang batch
config.gradient_checkpointing = True

model = PhoBERTAdvanced.from_pretrained(
    MODEL_NAME, config=config, num_samples=5, dropout_rate=0.3
).to(device)

# Bat gradient checkpointing tren encoder
model.roberta.encoder.gradient_checkpointing = True

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model    : PhoBERTAdvanced (CLS+Mean Pooling, MLP Head)')
print(f'Params   : {total:,} total | {trainable:,} trainable')
print(f'Pooling  : [CLS](768) + Mean(768) = 1536 dim')
print(f'Head     : Linear(1536->512, GELU) + MultiDropout(5x) + Linear(512->3)')

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.5, label_smoothing=0.1):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                             reduction='none', label_smoothing=self.label_smoothing)
        pt = F.softmax(logits, dim=1).gather(1, targets.unsqueeze(1)).squeeze(1)
        return (((1 - pt) ** self.gamma) * ce).mean()


class FGM:
    def __init__(self, model, epsilon=0.5):
        self.model   = model
        self.epsilon = epsilon
        self.backup  = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and 'word_embeddings' in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    param.data.add_(self.epsilon * param.grad / norm)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


class AWP:
    def __init__(self, model, adv_lr=0.01, adv_eps=0.001):
        self.model   = model
        self.adv_lr  = adv_lr
        self.adv_eps = adv_eps
        self.backup  = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    delta = self.adv_lr * param.grad / (norm + 1e-8)
                    param.data = torch.clamp(
                        param.data + delta,
                        self.backup[name] - self.adv_eps,
                        self.backup[name] + self.adv_eps
                    )

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


class EMA:
    def __init__(self, model, decay=0.999):
        self.model  = model
        self.decay  = decay
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = self.decay * self.shadow[n] + (1 - self.decay) * p.data

    def apply_shadow(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data         = self.shadow[n]

    def restore(self):
        for n, p in self.model.named_parameters():
            if n in self.backup:
                p.data = self.backup[n]
        self.backup = {}

    def save(self, path):
        torch.save({n: t.cpu() for n, t in self.shadow.items()}, path)

    def load(self, path, device):
        saved = torch.load(path, map_location=device)
        for n in self.shadow:
            if n in saved:
                self.shadow[n] = saved[n].to(device)


def rdrop_loss(logits1, logits2, labels, criterion, alpha=5.0):
    ce  = (criterion(logits1, labels) + criterion(logits2, labels)) / 2
    p   = F.log_softmax(logits1, dim=-1)
    q   = F.log_softmax(logits2, dim=-1)
    kl  = (F.kl_div(p, F.softmax(logits2, dim=-1), reduction='batchmean') +
           F.kl_div(q, F.softmax(logits1, dim=-1), reduction='batchmean')) / 2
    return ce + alpha * kl


print('Helpers defined: FocalLoss, FGM, AWP, EMA, R-Drop')

In [ ]:
class MetricsCallback(TrainerCallback):
    def __init__(self, log_file):
        self.log_file         = log_file
        self.train_loss_steps = []
        self.eval_records     = []
        self._load_history()

    def _load_history(self):
        if os.path.exists(self.log_file):
            with open(self.log_file, 'r') as f:
                data = json.load(f)
            self.train_loss_steps = [tuple(x) for x in data.get('train_loss_steps', [])]
            self.eval_records     = [tuple(x) for x in data.get('eval_records',     [])]
            print(f'Metrics history loaded: {len(self.train_loss_steps)} loss pts | {len(self.eval_records)} eval pts')

    def _save_history(self):
        with open(self.log_file, 'w') as f:
            json.dump({'train_loss_steps': self.train_loss_steps,
                       'eval_records':     self.eval_records}, f)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        if 'loss'    in logs: self.train_loss_steps.append((state.global_step, logs['loss']))
        if 'eval_f1' in logs: self.eval_records.append((state.global_step, logs['eval_accuracy'], logs['eval_f1']))

    def on_save(self, args, state, control, **kwargs):
        self._save_history()


class SaveEMACallback(TrainerCallback):
    def __init__(self, ema_file, trainer_ref):
        self.ema_file    = ema_file
        self.trainer_ref = trainer_ref

    def on_save(self, args, state, control, **kwargs):
        if self.trainer_ref._ema is not None:
            self.trainer_ref._ema.save(self.ema_file)


metrics_cb = MetricsCallback(log_file=METRICS_LOG_FILE)
print('Callbacks defined')

In [ ]:
class V4Trainer(Trainer):
    def __init__(self, *args,
                 class_weights=None,
                 focal_gamma=2.5,    label_smoothing=0.1,
                 fgm_epsilon=0.5,
                 awp_lr=0.01,        awp_eps=0.001,    awp_start_epoch=2,
                 rdrop_alpha=5.0,
                 ema_decay=0.999,    ema_state_file=None,
                 llrd_decay=0.95,
                 resumed_steps=0,
                 **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights   = class_weights
        self.focal_gamma     = focal_gamma
        self.label_smoothing = label_smoothing
        self.fgm_epsilon     = fgm_epsilon
        self.awp_lr          = awp_lr
        self.awp_eps         = awp_eps
        self.awp_start_epoch = awp_start_epoch
        self.rdrop_alpha     = rdrop_alpha
        self.ema_decay       = ema_decay
        self.ema_state_file  = ema_state_file
        self.llrd_decay      = llrd_decay
        self._steps          = resumed_steps
        self._focal = self._fgm = self._awp = self._ema = None

    def _lazy_init(self):
        if self._focal is not None:
            return
        w           = self.class_weights.to(self.model.device) if self.class_weights is not None else None
        self._focal = FocalLoss(alpha=w, gamma=self.focal_gamma, label_smoothing=self.label_smoothing)
        self._fgm   = FGM(self.model, epsilon=self.fgm_epsilon)
        self._awp   = AWP(self.model, adv_lr=self.awp_lr, adv_eps=self.awp_eps)
        self._ema   = EMA(self.model, decay=self.ema_decay)
        if self.ema_state_file and os.path.exists(self.ema_state_file):
            self._ema.load(self.ema_state_file, self.model.device)
            print(f'EMA state loaded: {self.ema_state_file}')
        else:
            print('V4Trainer initialized (fresh EMA)')

    def _fwd(self, model, inputs):
        return model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask']).logits

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        self._lazy_init()
        labels = inputs['labels']
        if model.training and self.rdrop_alpha > 0:
            l1   = self._fwd(model, inputs)
            l2   = self._fwd(model, inputs)
            loss = rdrop_loss(l1, l2, labels, self._focal, self.rdrop_alpha)
            out  = SequenceClassifierOutput(loss=loss, logits=l1)
        else:
            logits = self._fwd(model, inputs)
            loss   = self._focal(logits, labels)
            out    = SequenceClassifierOutput(loss=loss, logits=logits)
        return (loss, out) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        self._lazy_init()
        model.train()
        inputs = self._prepare_inputs(inputs)
        ga     = self.args.gradient_accumulation_steps

        # Standard forward + backward
        loss = self.compute_loss(model, inputs)
        self.accelerator.backward(loss / ga if ga > 1 else loss)

        # FGM: moi step (nhe, chi embedding)
        self._fgm.attack()
        loss_fgm = self.compute_loss(model, inputs)
        self.accelerator.backward(loss_fgm / ga if ga > 1 else loss_fgm)
        self._fgm.restore()

        self._steps += 1

        # AWP: full strength moi step (sau epoch warm-up)
        steps_per_epoch = max(1, len(self.train_dataset) //
                              (self.args.per_device_train_batch_size * max(1, ga)))
        if self._steps >= self.awp_start_epoch * steps_per_epoch:
            self._awp.attack()
            loss_awp = self.compute_loss(model, inputs)
            self.accelerator.backward(loss_awp / ga if ga > 1 else loss_awp)
            self._awp.restore()

        self._ema.update()
        return loss.detach()

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix='eval'):
        if self._ema is not None: self._ema.apply_shadow()
        result = super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)
        if self._ema is not None: self._ema.restore()
        return result

    def predict(self, test_dataset, ignore_keys=None, metric_key_prefix='test'):
        if self._ema is not None: self._ema.apply_shadow()
        result = super().predict(test_dataset, ignore_keys, metric_key_prefix)
        if self._ema is not None: self._ema.restore()
        return result

    def create_optimizer(self):
        """
        LLRD voi 3 nhom lr:
        - MLP head (proj + out_proj): base_lr * 10  -- layer moi, can hoc nhanh
        - Classifier dropouts       : base_lr * 5
        - Transformer layers 0-11   : base_lr * (0.95 ^ (11-i))
        - Embeddings                : base_lr * (0.95 ^ 12)
        """
        base_lr  = self.args.learning_rate
        no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
        groups   = []

        def _add(params_named, lr):
            wd_params  = [p for n, p in params_named if not any(nd in n for nd in no_decay)]
            nwd_params = [p for n, p in params_named if     any(nd in n for nd in no_decay)]
            if wd_params:  groups.append({'params': wd_params,  'weight_decay': self.args.weight_decay, 'lr': lr})
            if nwd_params: groups.append({'params': nwd_params, 'weight_decay': 0.0,                   'lr': lr})

        # MLP head: lr cao nhat vi la layer moi khoi tao
        _add([(n, p) for n, p in self.model.named_parameters()
              if any(k in n for k in ['proj', 'out_proj', 'act'])], base_lr * 10)

        # Dropout layers
        _add([(n, p) for n, p in self.model.named_parameters()
              if 'dropouts' in n], base_lr * 5)

        # Transformer layers: LLRD
        for i in range(11, -1, -1):
            lr_i = base_lr * (self.llrd_decay ** (11 - i))
            _add([(n, p) for n, p in self.model.named_parameters()
                  if f'layer.{i}.' in n], lr_i)

        # Embeddings: lr thap nhat
        _add([(n, p) for n, p in self.model.named_parameters()
              if 'embeddings' in n and 'layer' not in n],
             base_lr * (self.llrd_decay ** 12))

        try:
            self.optimizer = AdamW(groups, lr=base_lr, betas=(0.9, 0.999), eps=1e-8, fused=True)
            print('Using fused AdamW')
        except TypeError:
            self.optimizer = AdamW(groups, lr=base_lr, betas=(0.9, 0.999), eps=1e-8)
            print('Using standard AdamW')

        emb_lr = base_lr * (self.llrd_decay ** 12)
        print(f'LLRD: mlp_head={base_lr*10:.1e} | layer11={base_lr:.1e} | emb={emb_lr:.1e}')
        return self.optimizer


def compute_metrics(eval_pred):
    preds  = np.argmax(eval_pred.predictions, axis=-1)
    labels = eval_pred.label_ids
    acc    = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}


print('V4Trainer and compute_metrics ready')

In [ ]:
# ---- Tim checkpoint moi nhat ----
def get_latest_checkpoint(ckpt_dir):
    ckpts = sorted(
        glob.glob(os.path.join(ckpt_dir, 'checkpoint-*')),
        key=lambda x: int(x.split('-')[-1])
    )
    return ckpts[-1] if ckpts else None

latest_ckpt   = get_latest_checkpoint(CHECKPOINT_DIR)
resumed_steps = int(latest_ckpt.split('-')[-1]) if latest_ckpt else 0

if latest_ckpt:
    print(f'Resuming from : {latest_ckpt}')
    print(f'Steps done    : {resumed_steps}')
    print(f'EMA state     : {"found" if os.path.exists(EMA_STATE_FILE) else "not found (EMA reinit)"}')
else:
    print('No checkpoint found — starting fresh')

# ---- Training config ----
# batch_size=32 + grad_accum=2: effective=64, it context switch hon
# Gradient checkpointing cho phep batch_size=32 on dinh tren T4 16GB
BATCH_SIZE = 32
GRAD_ACCUM = 2
LR         = 2e-5
NUM_EPOCHS = 10

training_args = TrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine_with_restarts',
    max_grad_norm               = 1.0,
    eval_strategy               = 'steps',
    eval_steps                  = 300,
    logging_steps               = 50,
    save_strategy               = 'steps',
    save_steps                  = 300,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    fp16                        = True,
    dataloader_num_workers      = 4,
    dataloader_pin_memory       = True,
    group_by_length             = True,
    gradient_checkpointing      = True,
    seed                        = SEED,
    report_to                   = 'none',
)

trainer = V4Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tk_train,
    eval_dataset    = tk_val,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    class_weights   = class_weights_tensor,
    focal_gamma     = 2.5,
    label_smoothing = 0.1,
    fgm_epsilon     = 0.5,
    awp_lr          = 0.01,
    awp_eps         = 0.001,
    awp_start_epoch = 2,
    rdrop_alpha     = 5.0,
    ema_decay       = 0.999,
    ema_state_file  = EMA_STATE_FILE,
    llrd_decay      = 0.95,
    resumed_steps   = resumed_steps,
    callbacks       = [
        EarlyStoppingCallback(early_stopping_patience=5),
        metrics_cb,
    ]
)
trainer.add_callback(SaveEMACallback(ema_file=EMA_STATE_FILE, trainer_ref=trainer))

print('=' * 55)
print('STARTING V4 TRAINING')
print('=' * 55)

t0           = time.time()
train_result = trainer.train(resume_from_checkpoint=latest_ckpt)
elapsed      = time.time() - t0

metrics_cb._save_history()
print(f'Training done: {elapsed/60:.1f} min | Train loss: {train_result.metrics.get("train_loss", 0):.4f}')

In [ ]:
pred_out    = trainer.predict(tk_test)
predictions = np.argmax(pred_out.predictions, axis=-1)
true_labels = pred_out.label_ids
m           = pred_out.metrics

acc  = m['test_accuracy']
prec = m['test_precision']
rec  = m['test_recall']
f1   = m['test_f1']

print(f'Test Accuracy : {acc*100:.2f}%')
print(f'Test Precision: {prec*100:.2f}%')
print(f'Test Recall   : {rec*100:.2f}%')
print(f'Test F1 macro : {f1*100:.2f}%')
print()
print(classification_report(true_labels, predictions,
                             target_names=['NEGATIVE', 'POSITIVE', 'NEUTRAL'], digits=4))

report_str = classification_report(true_labels, predictions,
                                    target_names=['NEGATIVE', 'POSITIVE', 'NEUTRAL'], digits=4)
with open(f'{OUTPUT_DIR}/classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(f'Accuracy : {acc*100:.2f}%\n')
    f.write(f'F1 macro : {f1*100:.2f}%\n\n')
    f.write(report_str)

In [ ]:
LABEL_LIST = ['NEGATIVE', 'POSITIVE', 'NEUTRAL']

cm      = confusion_matrix(true_labels, predictions)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
prec_per, rec_per, f1_per, _ = precision_recall_fscore_support(
    true_labels, predictions, average=None, zero_division=0
)

fig = plt.figure(figsize=(16, 18))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# Row 0: Confusion matrices
ax_cm  = fig.add_subplot(gs[0, 0])
ax_cmn = fig.add_subplot(gs[0, 1])

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
            xticklabels=LABEL_LIST, yticklabels=LABEL_LIST, annot_kws={'size': 13})
ax_cm.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax_cm.set_xlabel('Predicted'); ax_cm.set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=ax_cmn,
            xticklabels=LABEL_LIST, yticklabels=LABEL_LIST, annot_kws={'size': 13})
ax_cmn.set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')
ax_cmn.set_xlabel('Predicted'); ax_cmn.set_ylabel('True')

# Row 1: Per-class + Overall
ax_pc = fig.add_subplot(gs[1, 0])
ax_ov = fig.add_subplot(gs[1, 1])

x = np.arange(3)
w = 0.25
for offset, vals, label, color in [
    (-w, prec_per, 'Precision', '#42A5F5'),
    (0,  rec_per,  'Recall',    '#66BB6A'),
    (w,  f1_per,   'F1',        '#FF7043')
]:
    bars = ax_pc.bar(x + offset, vals, w, label=label, color=color)
    for bar in bars:
        ax_pc.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                   f'{bar.get_height():.3f}', ha='center', fontsize=8)
ax_pc.set_xticks(x); ax_pc.set_xticklabels(LABEL_LIST)
ax_pc.set_ylim(0, 1.15); ax_pc.set_ylabel('Score')
ax_pc.set_title('Per-class Metrics', fontsize=14, fontweight='bold')
ax_pc.legend(loc='upper right')

bars_ov = ax_ov.bar(
    ['Accuracy', 'Precision\n(macro)', 'Recall\n(macro)', 'F1\n(macro)'],
    [acc, prec, rec, f1],
    color=['#5C6BC0', '#42A5F5', '#66BB6A', '#FF7043'],
    width=0.55, edgecolor='#333'
)
for bar, val in zip(bars_ov, [acc, prec, rec, f1]):
    ax_ov.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
               f'{val*100:.2f}%', ha='center', fontweight='bold', fontsize=12)
ax_ov.set_ylim(0, 1.15); ax_ov.set_ylabel('Score')
ax_ov.set_title('Overall Metrics (Test Set)', fontsize=14, fontweight='bold')
ax_ov.axhline(0.90, color='orange', linestyle='--', alpha=0.6, label='Target 90%')
ax_ov.axhline(0.95, color='red',    linestyle='--', alpha=0.4, label='Target 95%')
ax_ov.legend()

# Row 2: Training curves
ax_loss = fig.add_subplot(gs[2, 0])
ax_f1c  = fig.add_subplot(gs[2, 1])

if metrics_cb.train_loss_steps:
    steps_l, losses = zip(*metrics_cb.train_loss_steps)
    ax_loss.plot(steps_l, losses, color='#EF5350', linewidth=1.2, label='Train Loss')
    ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('Loss')
    ax_loss.set_title('Training Loss Curve', fontsize=14, fontweight='bold')
    ax_loss.legend(); ax_loss.grid(alpha=0.3)

if metrics_cb.eval_records:
    steps_e, accs_e, f1s_e = zip(*metrics_cb.eval_records)
    ax_f1c.plot(steps_e, [v*100 for v in accs_e], color='#42A5F5',
                linewidth=1.5, marker='o', ms=4, label='Val Accuracy')
    ax_f1c.plot(steps_e, [v*100 for v in f1s_e],  color='#FF7043',
                linewidth=1.5, marker='s', ms=4, label='Val F1 macro')
    ax_f1c.axhline(90, color='orange', linestyle='--', alpha=0.6, label='Target 90%')
    ax_f1c.set_xlabel('Step'); ax_f1c.set_ylabel('Score (%)')
    ax_f1c.set_title('Validation Metrics Curve', fontsize=14, fontweight='bold')
    ax_f1c.legend(); ax_f1c.grid(alpha=0.3)

fig.suptitle('PhoBERT V4 Advanced - Training & Evaluation Summary', fontsize=18, fontweight='bold')
plt.savefig(f'{OUTPUT_DIR}/evaluation_summary_v4.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Charts saved to {OUTPUT_DIR}/evaluation_summary_v4.png')

In [ ]:
if trainer._ema is not None and not trainer._ema.backup:
    trainer._ema.apply_shadow()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

if trainer._ema is not None:
    trainer._ema.restore()

config_out = {
    'version'   : 'V4-Advanced',
    'base_model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'num_labels': NUM_LABELS,
    'label_map' : LABEL_NAMES,
    'architecture': {
        'pooling'    : 'CLS + Mean (1536 dim)',
        'head'       : 'Linear(1536->512, GELU) + MultiDropout(5x) + Linear(512->3)',
    },
    'techniques': [
        'cls_mean_pooling_1536', 'mlp_head_gelu_512',
        'multi_sample_dropout_5x_0.3', 'focal_loss_gamma2.5',
        'label_smoothing_0.1', 'class_weights',
        'FGM_eps0.5', 'AWP_lr0.01_eps0.001_full',
        'R-Drop_alpha5.0', 'EMA_decay0.999',
        'LLRD_decay0.95_mlphead10x', 'cosine_with_restarts',
        'early_stopping_patience5', 'gradient_checkpointing',
        'dynamic_padding_pad8', 'group_by_length', 'fused_adamw',
    ],
    'training'  : {
        'epochs': NUM_EPOCHS, 'batch_size': BATCH_SIZE,
        'grad_accum': GRAD_ACCUM, 'effective_batch': BATCH_SIZE * GRAD_ACCUM, 'lr': LR,
    },
    'data'      : {
        'total': len(df), 'train': len(df_train), 'val': len(df_val), 'test': len(df_test),
    },
    'results'   : {
        'test_accuracy' : round(acc,  4),
        'test_precision': round(prec, 4),
        'test_recall'   : round(rec,  4),
        'test_f1_macro' : round(f1,   4),
    }
}

with open(f'{OUTPUT_DIR}/training_config.json', 'w', encoding='utf-8') as f:
    json.dump(config_out, f, indent=2, ensure_ascii=False)

print(f'Model saved to: {OUTPUT_DIR}')
for fn in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, fn)) / (1024 * 1024)
    print(f'  {fn}: {size_mb:.2f} MB')
print('DONE')